# ETL Silver ➜ Gold ➜ FIFA 21 Players

Este notebook realiza a transformação dos dados da camada **Silver**
para a camada **Gold**, aplicando **modelagem dimensional (Star Schema)**.

### Tabelas Gold geradas:
- dim_ply (Jogador)
- dim_tm (Time)
- dim_pos (Posição)
- fat_ply_stats (Fato de atributos e ratings)

### Premissas:
- Silver já está limpa e tipada
- Chaves surrogate são geradas na Gold
- Uma linha na fato representa:
  **Jogador + Time + Posição**


## Imports e Configurações

In [1]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv("../.env")

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST'),
    'port': int(os.getenv('POSTGRES_PORT')),
    'database': os.getenv('POSTGRES_DB'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD')
}

DATABASE_URL = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

## Leitura da Silver

In [3]:
QUERY_SILVER = """
SELECT *
FROM silver.fifa21_players
"""

df_silver = pd.read_sql(QUERY_SILVER, engine)

df_silver.head()

,player_id,long_name,name,nationality,positions,age,overall_rating,potential_rating,team,contract_start_year,...,attack_work_rate,defense_work_rate,international_reputation,pace,shooting,passing,dribbling_stat,defending_stat,physical,hits
0,158023,Lionel Messi,L. Messi,Argentina,RW ST CF,33,93,93,FC Barcelona,2004.0,...,Medium,Low,5,85,92,91,95,38,65,372
1,20801,C. Ronaldo dos Santos Aveiro,Cristiano Ronaldo,Portugal,ST LW,35,92,92,Juventus,2018.0,...,High,Low,5,89,93,81,89,35,77,344
2,200389,Jan Oblak,J. Oblak,Slovenia,GK,27,91,93,Atlético Madrid,2014.0,...,Medium,Medium,3,87,92,78,90,52,90,86
3,192985,Kevin De Bruyne,K. De Bruyne,Belgium,CAM CM,29,91,91,Manchester City,2015.0,...,High,High,4,76,86,93,88,64,78,163
4,190871,Neymar da Silva Santos Jr.,Neymar Jr,Brazil,LW CAM,28,91,91,Paris Saint-Germain,2017.0,...,High,Medium,5,91,85,86,94,36,59,273


## Dimensões

### Dimensão Jogador (dim_ply)

In [4]:
dim_ply = (
    df_silver[[
        "player_id",
        "long_name",
        "name",
        "age",
        "height_cm",
        "weight_kg",
        "preferred_foot",
        "weak_foot",
        "skill_moves",
        "international_reputation",
        "nationality"
    ]]
    .drop_duplicates()
)

dim_ply.insert(0, "ply_key", range(1, len(dim_ply) + 1))

### Dimensão Time (dim_tm)

In [5]:
dim_tm = (
    df_silver[[
        "team",
        "contract_start_year",
        "contract_end_year",
        "joined_date"
    ]]
    .drop_duplicates()
)

dim_tm["joined_date"] = pd.to_datetime(dim_tm["joined_date"])

dim_tm.insert(0, "tm_key", range(1, len(dim_tm) + 1))

### Dimensão Posição (dim_pos)

In [6]:
dim_pos = (
    df_silver[[
        "positions",
        "best_position"
    ]]
    .drop_duplicates()
)

dim_pos.insert(0, "pos_key", range(1, len(dim_pos) + 1))

## Tabela Fato

### Mapeamento de Chaves (Lookups)

In [7]:
df_fact = df_silver.copy()

df_fact = df_fact.merge(
    dim_ply[["ply_key", "player_id"]],
    on="player_id",
    how="left"
)

df_fact = df_fact.merge(
    dim_tm[["tm_key", "team", "contract_start_year", "contract_end_year"]],
    on=["team", "contract_start_year", "contract_end_year"],
    how="left"
)

df_fact = df_fact.merge(
    dim_pos[["pos_key", "positions", "best_position"]],
    on=["positions", "best_position"],
    how="left"
)


### Construção da fat_ply_stats

In [8]:
fat_ply_stats = df_fact[[
    "ply_key",
    "tm_key",
    "pos_key",

    "overall_rating",
    "potential_rating",
    "best_overall_rating",
    "growth",
    "total_stats",
    "base_stats",
    "hits",

    "value_eur",
    "wage_eur",
    "release_clause_eur",

    "pace",
    "shooting",
    "passing",
    "dribbling_stat",
    "defending_stat",
    "physical",

    "attacking_total",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",

    "skill_total",
    "dribbling",
    "curve",
    "fk_accuracy",
    "long_passing",
    "ball_control",

    "movement_total",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",

    "power_total",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",

    "mentality_total",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "composure",

    "defending_total",
    "marking",
    "standing_tackle",
    "sliding_tackle",

    "goalkeeping_total",
    "gk_diving",
    "gk_handling",
    "gk_kicking",
    "gk_positioning",
    "gk_reflexes"
]].rename(columns={
    "ply_key": "ply_srk",
    "tm_key": "tm_srk",
    "pos_key": "pos_srk"
})

fat_ply_stats.insert(0, "fts_key", range(1, len(fat_ply_stats) + 1))

### Validações Básicas

In [9]:
assert fat_ply_stats["ply_srk"].isnull().sum() == 0
assert fat_ply_stats["tm_srk"].isnull().sum() == 0
assert fat_ply_stats["pos_srk"].isnull().sum() == 0

## Escrita das tabelas Gold no banco

In [10]:
dim_ply.to_sql(
    "dim_ply",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

dim_tm.to_sql(
    "dim_tm",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

dim_pos.to_sql(
    "dim_pos",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

fat_ply_stats.to_sql(
    "fat_ply_stats",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)


ProgrammingError: (psycopg2.errors.InvalidSchemaName) schema "gold" does not exist
LINE 2: CREATE TABLE gold.dim_ply (
                     ^

[SQL: 
CREATE TABLE gold.dim_ply (
	ply_key BIGINT, 
	player_id BIGINT, 
	long_name TEXT, 
	name TEXT, 
	age BIGINT, 
	height_cm FLOAT(53), 
	weight_kg FLOAT(53), 
	preferred_foot TEXT, 
	weak_foot BIGINT, 
	skill_moves BIGINT, 
	international_reputation BIGINT, 
	nationality TEXT
)

]
(Background on this error at: https://sqlalche.me/e/20/f405)

## Exportação CSV da Gold Fato

In [ ]:
OUTPUT_DIR = "../Data Layer/silver"
OUTPUT_FILE_CSV = "fifa21_gold.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_CSV)

fat_ply_stats.to_csv(output_path, index=False)

print(f"CSV Gold gerado em: {output_path}")

## Validação pós-carga

In [ ]:
pd.read_sql(
    "SELECT COUNT(*) FROM gold.fat_ply_stats",
    engine
)